# 1 · Set Up the FABRIC Slice

Provision a 3-node slice (Alice / BMv2 switch / Bob), upload QFabric, install BMv2,
compile the P4 quantum-channel program, start the switch, and wire the data-plane network.

This notebook is a thin wrapper over the tested functions in `scripts/deploy_fabric.py`
(`create_slice`, `upload_project`, `install_deps`, `configure_switch`, `setup_dataplane_ips`),
so it stays in lock-step with the command-line deployer.

**Prerequisite:** `00_overview` (env check passed, FABRIC tokens configured).
After this notebook the slice is live and the data plane is running — go to `02_run_experiment`.

### At a glance
- **Purpose:** stand up the 3-node slice and bring the quantum data plane online.
- **Inputs:** `SLICE_NAME`, the three FABRIC sites, and `SCENARIO` (sets the initial loss threshold).
- **Outputs:** a live slice — BMv2 running on the switch with the fiber-loss + MAC-rewrite tables, and data-plane IPs (10.10.1.x) on Alice/Bob for the classical channel.
- **Runs on / runtime:** FABRIC JupyterHub; **~10–20 min** with the source build, or **~3–5 min** if you set `BMV2_IMAGE` (prebuilt container pull).
- **If something fails:** BMv2 build errors are almost always a missing apt dep on the switch — the error names it; add it to `scripts/install_bmv2.sh`. Re-running this notebook is safe (it reuses an existing slice and preserves the node venvs).

## Configuration

In [7]:
SLICE_NAME = 'qfabric-bb84-2'
SITE_ALICE = 'TACC'      # Alice site
SITE_BOB   = 'TACC'      # Bob site (different site → real WAN classical channel)
SITE_SW    = 'TACC'      # BMv2 switch (colocated with Alice)
SCENARIO   = 'validation/scenarios/fabric_1km.yml'   # repo-relative; sizes the loss model

# Optional: run BMv2 from a PREBUILT container instead of compiling it on the
# switch (saves the multi-minute source build). Set to the published image
# 'ghcr.io/kthare10/qfabric-bmv2:latest' (or 'p4lang/p4c:latest'); leave '' to
# build BMv2 from source on the switch.
BMV2_IMAGE = 'ghcr.io/kthare10/qfabric-bmv2:latest'

In [8]:
import sys
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

# deploy_fabric.py holds the tested provisioning/run logic; the notebooks are
# thin wrappers around it so there is a single source of truth.
import deploy_fabric as df
from qne.config import ScenarioConfig

fablib = df.get_fablib()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/Users/kthare10/work/fabric_config/id_token.json
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
Bastion Host,bastion.fabric-testbed.net
Bastion Username,kthare10_0011904101
Bastion Private Key File,/Users/kthare10/.ssh/bastion-prod-2
Slice Public Key File,/Users/kthare10/.ssh/id_rsa.pub
Slice Private Key File,/Users/kthare10/.ssh/id_rsa


## Step 1 — Provision the slice (Alice, Bob, switch + L2 networks)
Submits the slice and waits for SSH. Takes a few minutes.

In [9]:
# Reuse the slice if it already exists (re-runnable); otherwise provision it.
try:
    slice_obj = fablib.get_slice(name=SLICE_NAME)
    print(f"Reusing existing slice '{SLICE_NAME}' (state: {slice_obj.get_state()})")
except Exception:
    slice_obj = df.create_slice(fablib, SLICE_NAME, SITE_ALICE, SITE_BOB, SITE_SW)

User: kthare10@email.unc.edu bastion key is valid!
Configuration is valid
Reusing existing slice 'qfabric-bb84-2' (state: StableOK)


## Step 2 — Upload QFabric and install dependencies
`upload_project` copies only the needed files (no `.venv`/`.git`). Then either:
- **source build** (`BMV2_IMAGE = ''`): `install_deps` sets up Alice/Bob Python deps and builds BMv2 on the switch (several minutes the first time); or
- **prebuilt image** (`BMV2_IMAGE` set): installs Alice/Bob deps, installs Docker on the switch and pulls the image, and enables the container path — no source build.

In [10]:
import os
df.upload_project(slice_obj)

if BMV2_IMAGE:
    os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE   # configure_switch uses the container
    df.install_deps(slice_obj, build_bmv2=False)    # Alice/Bob Python deps only
    df.setup_switch_docker(slice_obj, BMV2_IMAGE)   # install Docker + pull image on switch
else:
    df.install_deps(slice_obj)                      # build BMv2 from source on the switch


=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)

=== Installing dependencies ===
  Installing Python deps on alice...
    alice: deps OK
  Installing Python deps on bob...
    bob: deps OK
  Skipping switch BMv2 source build (using prebuilt Docker image).

=== Preparing switch to run BMv2 from Docker image: ghcr.io/kthare10/qfabric-bmv2:latest ===
=== QFabric: preparing switch to run BMv2 from Docker image ===
  Image: ghcr.io/kthare10/qfabric-bmv2:latest
--- Docker already installed ---
--- Pulling ghcr.io/kthare10/qfabric-bmv2:latest (one-time; cached thereafter) ---
latest: Pulling from kthare10/qfabric-bmv2
Digest: sha256:217abcecb0e1cca16bd84922a3982dacb9c117c7f36d03051b3b8e4f15fb46bb
Status: Image is up to date for ghcr.io/kthare10/qfabric-bmv2:latest
ghcr.io/kthare10/qfabric-bmv2:latest
--- Verifying the image has the BMv2 toolchain ---
1.15.3-unk

## Step 3 — Compile P4, start the switch, and set up the data plane
Computes the fiber-loss threshold from the scenario, starts BMv2 (durable `systemd-run`)
with the loss-model + MAC-rewrite tables, and assigns data-plane IPs so the classical
channel rides the FABRIC L2 link (the management network blocks cross-site TCP).

In [6]:
threshold = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO).loss_threshold_u32
print(f'P4 loss threshold (u32): {threshold}')

alice_mac, bob_mac, sw_alice_mac, sw_bob_mac, _, _ = df.configure_switch(slice_obj, threshold)
alice_ip, bob_ip = df.setup_dataplane_ips(slice_obj, alice_mac, bob_mac)
print('\nSlice ready: switch running, data plane up.')

P4 loss threshold (u32): 193305371

=== Configuring switch (threshold=193305371) ===
  Switch interfaces: enp8s0 (Alice), enp7s0 (Bob)
  Alice MAC: 22:F5:5F:CE:C3:60
  Bob MAC:   12:E7:E6:F5:75:0C
  Switch Alice-side MAC: 3A:84:1C:A5:5F:9C
  Switch Bob-side MAC:   3A:22:D7:38:CD:CD
  Using BMv2 Docker image: ghcr.io/kthare10/qfabric-bmv2:latest
  Compiling P4 (in container)...
  Starting BMv2 (container: --privileged --network host)...
  Configuring tables...
  Switch configured and running

=== Setting up data-plane IPs ===
  Alice: 10.10.1.1/24 on enp7s0
  Bob:   10.10.1.2/24 on enp7s0
  Adding static ARP entries...
  ARP: Alice → 10.10.1.2 via 3a:84:1c:a5:5f:9c (switch Alice-side)
  ARP: Bob → 10.10.1.1 via 3a:22:d7:38:cd:cd (switch Bob-side)
  Testing connectivity (ping)...
  Ping successful!

Slice ready: switch running, data plane up.


---
**Next:** `02_run_experiment`.